# Week 10: Transparency, Interpretability & Limitations

## Strategy: Transparent Scaled Ensemble
For our 10th round (19 data points), the focus shifts to **Transparency**. 
We continue using our robust `Scaled Hybrid Ensemble` (20 NNs + 1 GP), but we are now explicitly extracting and logging the internal decision metrics:
1. **Predicted Mean (Exploitation)**
2. **Predictive Standard Deviation (Exploration/Uncertainty)**
3. **Acquisition Value (UCB)**

This makes the decision-making process of our "black-box" surrogate models interpretable and reproducible.

In [1]:
import numpy as np
import warnings
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import sys
import os

# Ensure we can import from src
sys.path.append(os.path.abspath('..'))
from src.utils import load_data

warnings.filterwarnings("ignore")
np.random.seed(50) # Week 10 Seed - strictly fixed for reproducibility

print("Ready for Transparent Optimization")

Ready for Transparent Optimization


In [2]:
def suggest_next_point_transparent(func_id, X_train, y_train):
    print(f"\n--- Optimizing Function {func_id} ---")
    
    # 1. Preprocessing
    scaler_x = StandardScaler()
    X_scaled = scaler_x.fit_transform(X_train)
    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    
    # 2. Model Definition (Scaled Ensemble from W9)
    hidden_layers = (128, 64) if func_id in [7, 8] else (64, 32)
    base_mlp = MLPRegressor(hidden_layer_sizes=hidden_layers, alpha=0.01, 
                            activation='tanh', solver='lbfgs', max_iter=2000)
    
    regr = BaggingRegressor(estimator=base_mlp, n_estimators=20, random_state=42, n_jobs=-1)
    regr.fit(X_scaled, y_scaled)
    
    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=0.1)
    gp_model = GaussianProcessRegressor(kernel=kernel, normalize_y=False)
    gp_model.fit(X_scaled, y_scaled)
    
    # 3. Transparent Objective Function
    best_idx = np.argmax(y_train)
    x_start_original = X_train[best_idx]
    x_start_scaled = scaler_x.transform(x_start_original.reshape(1, -1)).flatten()
    
    # We will store the metrics of the best found point here
    decision_metrics = {}

    def objective_function(x):
        x_reshaped = x.reshape(1, -1)
        
        nn_preds = np.array([est.predict(x_reshaped)[0] for est in regr.estimators_])
        avg_nn, std_nn = np.mean(nn_preds), np.std(nn_preds)
        
        gp_pred, gp_std = gp_model.predict(x_reshaped, return_std=True)
        gp_pred, gp_std = gp_pred[0], gp_std[0]
        
        comb_mean = 0.6 * avg_nn + 0.4 * gp_pred
        comb_std = 0.6 * std_nn + 0.4 * gp_std
        
        ucb = comb_mean + 1.96 * comb_std
        dist_sq = np.sum((x_reshaped - x_start_scaled)**2)
        penalty = 10.0 * np.exp(-dist_sq / (2 * 0.1**2))
        
        # Save transparent metrics
        decision_metrics['mean'] = comb_mean
        decision_metrics['std'] = comb_std
        decision_metrics['ucb'] = ucb
        decision_metrics['penalty'] = penalty
        
        return -ucb + penalty

    # 4. Trust Region bounds
    radius = 0.2
    bounds_scaled = []
    for i in range(X_train.shape[1]):
        mean, scale = scaler_x.mean_[i], scaler_x.scale_[i]
        curr_val = x_start_original[i]
        lower = (max(0.0, curr_val - radius) - mean) / scale
        upper = (min(1.0, curr_val + radius) - mean) / scale
        bounds_scaled.append((lower, upper))
    
    x_init = x_start_scaled + np.random.uniform(-0.1, 0.1, size=x_start_scaled.shape)
    
    res = minimize(fun=objective_function, x0=x_init, method='L-BFGS-B', 
                   bounds=bounds_scaled, options={'maxiter': 100})
    
    next_point = np.clip(scaler_x.inverse_transform(res.x.reshape(1, -1)).flatten(), 0.0, 1.0)
    
    # Transparency Log
    print(f"   [Decision Log] Surrogate Predicted Scaled Mean: {decision_metrics['mean']:.4f}")
    print(f"   [Decision Log] Ensemble Uncertainty (Std): {decision_metrics['std']:.4f}")
    print(f"   [Decision Log] Repulsion Penalty Applied: {decision_metrics['penalty']:.4f}")
    
    return next_point

In [3]:
submission_queries = {}
print("Starting Round 10 Evaluation...")

for func_id in range(1, 9):
    # Ensure you have updated data to 19 points before running
    X_known, y_known = load_data(func_id)
    next_x = suggest_next_point_transparent(func_id, X_known, y_known)
    submission_queries[func_id] = next_x

print("\n" + "="*30)
print("FORMATTED SUBMISSION OUTPUT")
print("="*30)

for func_id, x_val in submission_queries.items():
    formatted_str = "-".join([f"{val:.6f}" for val in x_val])
    print(f"function_number: {func_id}: {formatted_str}")

   [Decision Log] Surrogate Predicted Scaled Mean: 1.5075
   [Decision Log] Ensemble Uncertainty (Std): 0.1674
   [Decision Log] Repulsion Penalty Applied: 0.0000

FORMATTED SUBMISSION OUTPUT
function_number: 1: 0.796656-0.533000
function_number: 2: 0.511417-0.727013
function_number: 3: 0.451546-0.774352-0.633932
function_number: 4: 0.220631-0.382944-0.443891-0.432171
function_number: 5: 1.000000-0.892112-1.000000-1.000000
function_number: 6: 0.141863-0.298622-0.796785-0.995638-0.000000
function_number: 7: 0.000000-0.360418-0.439838-0.027773-0.156875-0.609856
function_number: 8: 0.000000-0.384962-0.000000-0.413207-0.274043-0.602089-0.000000-0.346610
